# Walkthrough: how the invoice checker works

The client (Pepsi) pays the ad agency (GroupM). GroupM buys ads from publishers (Facebook, Instagram, ...).
Each month the publisher sends a bill. For every **bill line** we check four things, then decide.

*Every step below has the same three parts: **① the input**, **② the blank form** it goes into, **③ the filled result**.
Click the **•••** next to a picture to see the code behind it. The dashboard is at the very bottom.*

In [1]:
import json
from decimal import Decimal

import pandas as pd
from IPython.display import HTML, display

from invoice_recon import display as vz
from invoice_recon.config import get_settings
from invoice_recon.describe import blank_form
from invoice_recon.loaders import load_reference_data

settings = get_settings()
DATA = settings.data_folder
ref = load_reference_data()


def show(*parts):
    display(HTML(vz.wrap(*parts)))


def blank(form, caption):
    rows = [list(r.values()) for r in blank_form(form)]
    head = list(blank_form(form)[0])
    show(
        f"<div style='font-weight:600;margin-top:10px'>{form.__name__}: {caption}</div>",
        vz.table(head, rows),
    )


show(
    vz.flow(
        [
            ("Four things", "ordered · delivered · allowed by contract · billed"),
            (
                "Check every bill line",
                "match it → work out what should be paid → find why it differs",
            ),
            ("Decision", "approve · pay less · dispute · ask a person"),
        ],
        ["info", "plain", "good"],
    )
)

---
# Step 1.1: The 5 blank forms
A **form** is an empty list of fields. Every record must fit its form, or it is refused. Money is always exact (never a rounded decimal).

In [2]:
from invoice_recon.models import ContractTerms, DeliveryRecord, InvoiceLine, PlanLine, Result

blank(PlanLine, "what was ordered (the media plan)")
blank(DeliveryRecord, "what was delivered (views per month)")
blank(ContractTerms, "what the contract allows")
blank(InvoiceLine, "what was billed (one line of a bill)")
blank(Result, "our answer for one bill line")

field,kind of value,required,rules,meaning
plan_line_id,text,yes,can't be empty,"Unique ID of this plan row, e.g. PL-001."
purchase_order_number,text,yes,can't be empty,"Covers a whole campaign at one publisher, not a single plan line."
publisher,text,yes,can't be empty,"Who runs the ad, e.g. Instagram."
description,text,yes,can't be empty,"What is being bought: campaign, publisher and ad type."
start_date,date,yes,-,First day the ad runs.
end_date,date,yes,-,Last day the ad runs. Not before start_date.
planned_impressions,whole number,yes,>= 0,Views ordered for the whole run.
price_per_1000_views,exact decimal,yes,"never a float, >= 0, at most 4 decimal places","Agreed price for every 1,000 views (CPM)."
planned_cost,exact decimal,yes,"never a float, >= 0, at most 2 decimal places","Must equal views x price per view, to the cent."
status,"one of: active, cancelled",yes,-,"Active, or cancelled from cancelled_effective_date."


field,kind of value,required,rules,meaning
plan_line_id,text,yes,can't be empty,Which plan line these views belong to.
publisher,text,yes,can't be empty,Must be the same publisher as on the plan line.
month,text,yes,month as YYYY-MM,Which month the views were delivered in.
publisher_count,whole number,yes,>= 0,"Views counted by the publisher (Instagram, Facebook)."
ad_server_count,whole number,yes,>= 0,Views counted by the ad agency's (GroupM's) own ad server.


field,kind of value,required,rules,meaning
publisher,text,yes,can't be empty,The publisher these rules apply to.
agency,text,yes,can't be empty,"The ad agency that pays, e.g. GroupM."
client,text,yes,can't be empty,"The brand the ads are for, e.g. Pepsi."
bill_on,"one of: publisher_count, ad_server_count",yes,-,Whose view count is used for billing.
allowed_difference_percent,exact decimal,yes,"never a float, >= 0, <= 100",How far the publisher's count may be above the ad server count.
pay_for_extra_views_up_to_percent,exact decimal,yes,"never a float, >= 0, <= 100",0 = extra views are never paid. 10 = up to 10% extra views are paid.


field,kind of value,required,rules,meaning
invoice_number,text,yes,can't be empty,"The bill's number, e.g. IG-INV-2026-0142."
invoice_line_number,whole number,yes,>= 1,"Line position on the bill, starting at 1."
publisher,text,yes,can't be empty,Who sent the bill.
service_month,text,yes,month as YYYY-MM,Which month this line charges for.
purchase_order_number,text,no,can't be empty,Missing on about a quarter of lines.
line_description,text,yes,can't be empty,"The publisher's own wording, e.g. ""PZS | Reels | JAN26""."
billed_impressions,whole number,yes,>= 0,Views the publisher is charging for.
rate_per_1000_views,exact decimal,yes,"never a float, >= 0, at most 4 decimal places","Price per 1,000 views used on the bill."
amount,exact decimal,yes,"never a float, >= 0, at most 2 decimal places",Money charged. May be wrong; checked later.
source_file,text,yes,can't be empty,The bill file this line was read from.


field,kind of value,required,rules,meaning
invoice_number,text,yes,can't be empty,The bill this result is for.
invoice_line_number,whole number,yes,>= 1,The line on that bill.
matched_plan_line_id,text,no,can't be empty,None when no plan line matches.
match_confidence,number,yes,">= 0, <= 1",How sure the match is: 1.0 = certain.
expected_amount,exact decimal,yes,"never a float, >= 0, at most 2 decimal places",What the ad agency (GroupM) should pay under the contract.
invoiced_amount,exact decimal,yes,"never a float, >= 0, at most 2 decimal places",What the publisher billed.
cause,"one of: none, over_delivery_billed, over_delivery_within_contract, over_delivery_beyond_contract, outside_flight_dates, duplicate_line, count_mismatch_beyond_tolerance, billed_on_wrong_count, cancelled_line_billed, math_error, rate_mismatch, no_matching_order, unexplained_difference",yes,-,"Why the bill differs, from the fixed list of causes."
evidence,list of text,no,-,"Short facts that prove the cause, each naming its source."
decision,"one of: approve, pay_less, dispute, send_to_human",yes,-,"What to do: approve, pay less, dispute or send to a human."


---
# Step 1.2: Load the clean files

In [3]:
show(
    vz.flow(
        [
            ("① Input", "media plan · delivery · contracts"),
            ("② Blank forms", "PlanLine · DeliveryRecord · ContractTerms"),
            ("③ Filled and checked", "every record fits, and the files agree with each other"),
        ],
        ["plain", "info", "good"],
    )
)

## ① Input: the files as stored (first rows)

In [4]:
plan_raw = pd.read_excel(DATA / "media_plan.xlsx", dtype=str, keep_default_na=False)
cols = [
    "plan_line_id",
    "purchase_order_number",
    "publisher",
    "description",
    "start_date",
    "end_date",
    "planned_impressions",
    "price_per_1000_views",
]
deliv_raw = pd.read_csv(DATA / "delivery" / "delivery_2026-01.csv", dtype=str)
contracts_raw = json.loads((DATA / "contracts.json").read_text())


def preview(df):
    rows = [[str(v).replace(" 00:00:00", "") for v in r] for r in df.values.tolist()]
    return vz.table(list(df.columns), rows)


show(
    "<b>media_plan.xlsx</b> (what was ordered)",
    preview(plan_raw[cols].head(3)),
    "<br><b>delivery_2026-01.csv</b> (what was delivered)",
    preview(deliv_raw.head(3)),
    "<br><b>contracts.json</b> (what the contract allows)",
    preview(pd.DataFrame(contracts_raw).head(2)),
)

## ② The blank forms: `PlanLine`, `DeliveryRecord`, `ContractTerms` (shown in step 1.1)

## ③ Filled: every row became a record, and the files agree with each other

In [5]:
show(
    vz.tiles(
        [
            ("plan lines", str(len(ref.plan_lines)), "good"),
            ("delivery records", str(len(ref.deliveries)), "good"),
            ("contracts", str(len(ref.contracts)), "good"),
            ("cross-file checks passed", "6 / 6", "good"),
        ]
    )
)
first = list(ref.plan_lines.values())[:3]
show(
    "<b>First 3 plan lines, now filled forms</b>",
    vz.table(
        ["id", "publisher", "ordered views", "price per 1,000", "planned cost"],
        [
            [
                p.plan_line_id,
                p.publisher,
                f"{p.planned_impressions:,}",
                vz.usd(p.price_per_1000_views),
                vz.usd(p.planned_cost),
            ]
            for p in first
        ],
        numeric=["ordered views", "price per 1,000", "planned cost"],
    ),
)

id,publisher,ordered views,"price per 1,000",planned cost
PL-001,Instagram,"3,824,000",$8.25,"$31,548.00"
PL-002,Instagram,"6,506,000",$9.50,"$61,807.00"
PL-003,Instagram,"4,286,000",$12.50,"$53,575.00"


---
# Step 1.4: Exact money
Every amount is rounded to whole cents, **half a cent goes up**. Each line is rounded first, then lines are added.

In [6]:
from invoice_recon.money import cost_of_views, to_cents

views, price = 1_475_215, Decimal("8.25")
exact = Decimal(views) * price / 1000
show(
    vz.flow(
        [
            ("Views × price", f"{views:,} views × ${price} per 1,000"),
            ("Exact", f"${exact:,}"),
            ("Round half up", f"${cost_of_views(views, price):,}"),
        ],
        ["plain", "info", "good"],
    )
)
show(
    "<b>Why half up?</b> It is what invoices use. Python's default rounds differently:",
    vz.table(
        ["exact amount", "our rule (half up)", "Python default (banker's)"],
        [
            [a, f"${to_cents(Decimal(a))}", f"${Decimal(a).quantize(Decimal('0.01'))}"]
            for a in ["10.125", "2.675", "1.005"]
        ],
    ),
)

exact amount,our rule (half up),Python default (banker's)
10.125,$10.13,$10.12
2.675,$2.68,$2.68
1.005,$1.01,$1.00


---
# Step 2.2: Read the Facebook bills

In [7]:
show(
    vz.flow(
        [
            ("① Input", "Facebook's CSV bill, with a Total row at the end"),
            ("② Blank form", "InvoiceLine"),
            ("③ Filled and checked", "14 clean lines; they add up to the Total row"),
        ],
        ["plain", "info", "good"],
    )
)

## ① Input: the January bill, as Facebook sends it
The grey last row is Facebook's own **Total**. A **blank PO** (purchase order number) is marked red.

In [8]:
bill_path = DATA / "invoices" / "facebook_2026-01.csv"
raw = pd.read_csv(bill_path, dtype=str, keep_default_na=False)
raw = raw.rename(columns={"CPM (cost per 1,000 impressions) (USD)": "Price per 1,000 (USD)"})
rows = [
    [
        vz.badge("blank", "bad") if (v == "" and c == "PO Reference" and r[0]) else v
        for c, v in zip(raw.columns, r, strict=True)
    ]
    for r in raw.values.tolist()
]
show(vz.table(list(raw.columns), rows, dim=[len(rows) - 1]))

Invoice ID,Campaign name,Ad set name,Reporting starts,Reporting ends,Impressions,"Price per 1,000 (USD)",Amount spent (USD),PO Reference
FBADS-26-01-7731,Pepsi_WildCherry_Spring_2026,FB Feed Video,2026-01-01,2026-01-31,4577110,10.25,46915.38,PO-5011
FBADS-26-01-7731,Pepsi_WildCherry_Spring_2026,FB Feed Image,2026-01-01,2026-01-31,3340206,7.25,24216.49,PO-5011
FBADS-26-01-7731,Pepsi_WildCherry_Spring_2026,In-Stream,2026-01-01,2026-01-31,4316721,7.75,33454.59,PO-5011
FBADS-26-01-7731,Pepsi_WildCherry_Spring_2026,Marketplace,2026-01-01,2026-01-31,3002694,10.50,31528.29,blank
FBADS-26-01-7731,DietPepsi_NewYear_2026,FB Feed Video,2026-01-01,2026-01-31,1847275,10.25,18934.57,PO-5012
FBADS-26-01-7731,DietPepsi_NewYear_2026,FB Feed Image,2026-01-01,2026-01-31,2910381,7.75,22555.45,PO-5012
FBADS-26-01-7731,DietPepsi_NewYear_2026,Marketplace,2026-01-01,2026-01-31,2322940,6.50,15099.11,PO-5012
,Total,,,,22317327,,192703.88,


## ② The blank form: `InvoiceLine` (shown in step 1.1)

## ③ Filled: 14 bill lines, one per row (January 7 + February 7)

In [9]:
from invoice_recon.readers import read_facebook_invoices

fb_lines = read_facebook_invoices(DATA / "invoices")
show(
    vz.table(
        [
            "bill",
            "line",
            "month",
            "campaign | ad set",
            "PO",
            "impressions",
            "price per 1,000",
            "amount",
        ],
        [
            [
                ln.invoice_number[-4:],
                ln.invoice_line_number,
                ln.service_month,
                ln.line_description.replace("_", " "),
                ln.purchase_order_number or vz.badge("blank", "bad"),
                f"{ln.billed_impressions:,}",
                vz.usd(ln.rate_per_1000_views),
                vz.usd(ln.amount),
            ]
            for ln in fb_lines
        ],
        numeric=["impressions", "price per 1,000", "amount"],
    )
)

checks = []
for name in ["facebook_2026-01.csv", "facebook_2026-02.csv"]:
    total = pd.read_csv(DATA / "invoices" / name, dtype=str, keep_default_na=False).iloc[-1]
    mine = [ln for ln in fb_lines if ln.source_file == name]
    ok = sum(ln.billed_impressions for ln in mine) == int(total["Impressions"]) and sum(
        ln.amount for ln in mine
    ) == Decimal(total["Amount spent (USD)"])
    checks.append(
        (
            f"{name[9:16]}: the 7 lines add up to Facebook's Total",
            "✓ yes" if ok else "✗ NO",
            "good" if ok else "bad",
        )
    )
show(vz.tiles(checks))

bill,line,month,campaign | ad set,PO,impressions,"price per 1,000",amount
7731,1,2026-01,Pepsi WildCherry Spring 2026 | FB Feed Video,PO-5011,"4,577,110",$10.25,"$46,915.38"
7731,2,2026-01,Pepsi WildCherry Spring 2026 | FB Feed Image,PO-5011,"3,340,206",$7.25,"$24,216.49"
7731,3,2026-01,Pepsi WildCherry Spring 2026 | In-Stream,PO-5011,"4,316,721",$7.75,"$33,454.59"
7731,4,2026-01,Pepsi WildCherry Spring 2026 | Marketplace,blank,"3,002,694",$10.50,"$31,528.29"
7731,5,2026-01,DietPepsi NewYear 2026 | FB Feed Video,PO-5012,"1,847,275",$10.25,"$18,934.57"
7731,6,2026-01,DietPepsi NewYear 2026 | FB Feed Image,PO-5012,"2,910,381",$7.75,"$22,555.45"
7731,7,2026-01,DietPepsi NewYear 2026 | Marketplace,PO-5012,"2,322,940",$6.50,"$15,099.11"
8094,1,2026-02,Pepsi WildCherry Spring 2026 | FB Feed Video,PO-5011,"4,148,629",$10.25,"$42,523.45"
8094,2,2026-02,Pepsi WildCherry Spring 2026 | FB Feed Image,blank,"2,979,452",$7.25,"$21,601.03"
8094,3,2026-02,Pepsi WildCherry Spring 2026 | In-Stream,blank,"3,880,677",$7.75,"$30,075.25"


---
# Step 3.1: Match each bill line to the plan line it belongs to
Facebook and GroupM word things differently, so we match with simple rules.

In [10]:
show(
    vz.flow(
        [
            ("① Input", "bill line: Facebook's wording + PO"),
            (
                "② Narrow to a campaign",
                "by the PO, or by the campaign wording (a PO covers a whole campaign)",
            ),
            (
                "③ Pick the plan line",
                "the one with the same ad type: feed video, feed image, in-stream, marketplace",
            ),
        ],
        ["plain", "info", "good"],
    )
)

In [11]:
from invoice_recon.matching import match_lines

matches = match_lines(fb_lines, ref)
rows = []
for ln, m in zip(fb_lines, matches, strict=True):
    plan = ref.plan_lines[m.plan_line_id]
    cancelled = vz.badge("cancelled order", "bad") if plan.status.value == "cancelled" else ""
    rows.append(
        [
            f"{ln.invoice_number[-4:]} #{ln.invoice_line_number}",
            ln.line_description.replace("_", " "),
            ln.purchase_order_number or vz.badge("blank", "bad"),
            "→",
            vz.Raw(f"<b>{plan.plan_line_id}</b> {cancelled}"),
            plan.description,
            vz.badge("PO + wording agree", "good")
            if m.confidence == 1
            else vz.badge("wording only", "info"),
        ]
    )
show(
    vz.table(
        ["bill line", "Facebook's wording", "PO", "", "plan line", "GroupM's wording", "how"], rows
    )
)
show(
    "<b>14 of 14 lines found their plan line.</b> Anything unclear is <b>not guessed</b>: "
    "it goes to a person."
)

bill line,Facebook's wording,PO,,plan line,GroupM's wording,how
7731 #1,Pepsi WildCherry Spring 2026 | FB Feed Video,PO-5011,→,PL-009,Pepsi Wild Cherry Spring - Facebook Feed video,PO + wording agree
7731 #2,Pepsi WildCherry Spring 2026 | FB Feed Image,PO-5011,→,PL-010 cancelled order,Pepsi Wild Cherry Spring - Facebook Feed image,PO + wording agree
7731 #3,Pepsi WildCherry Spring 2026 | In-Stream,PO-5011,→,PL-011,Pepsi Wild Cherry Spring - Facebook In-stream video,PO + wording agree
7731 #4,Pepsi WildCherry Spring 2026 | Marketplace,blank,→,PL-012,Pepsi Wild Cherry Spring - Facebook Marketplace image,wording only
7731 #5,DietPepsi NewYear 2026 | FB Feed Video,PO-5012,→,PL-013,Diet Pepsi New Year Refresh - Facebook Feed video,PO + wording agree
7731 #6,DietPepsi NewYear 2026 | FB Feed Image,PO-5012,→,PL-014,Diet Pepsi New Year Refresh - Facebook Feed image,PO + wording agree
7731 #7,DietPepsi NewYear 2026 | Marketplace,PO-5012,→,PL-016,Diet Pepsi New Year Refresh - Facebook Marketplace image,PO + wording agree
8094 #1,Pepsi WildCherry Spring 2026 | FB Feed Video,PO-5011,→,PL-009,Pepsi Wild Cherry Spring - Facebook Feed video,PO + wording agree
8094 #2,Pepsi WildCherry Spring 2026 | FB Feed Image,blank,→,PL-010 cancelled order,Pepsi Wild Cherry Spring - Facebook Feed image,wording only
8094 #3,Pepsi WildCherry Spring 2026 | In-Stream,blank,→,PL-011,Pepsi Wild Cherry Spring - Facebook In-stream video,wording only


---
# Step 4.1: What should GroupM pay? (the contract math)

In [12]:
from invoice_recon.contract_math import payable

fb = ref.contracts["Facebook"]
tol = fb.allowed_difference_percent
extra = fb.pay_for_extra_views_up_to_percent
show(
    vz.card(
        "Facebook's contract",
        [
            "Bills on the publisher's own view count",
            f"...unless it is more than {tol}% above GroupM's ad server count",
            f"Never pay for more than the month's planned views (+{extra}%)",
        ],
        "info",
    ),
    vz.flow(
        [
            ("1. Pick the count", "publisher's or ad server's"),
            ("2. Cap it", "at the month's planned views"),
            ("3. × the plan's price", "per 1,000 views, rounded half up"),
        ],
        ["plain", "plain", "plain"],
    ),
)
pays = []
for ln, m in zip(fb_lines, matches, strict=True):
    plan = ref.plan_lines[m.plan_line_id]
    pays.append(
        payable(plan, ref.delivery_for(plan.plan_line_id, ln.service_month), fb, ln.service_month)
    )

## One line in detail: January, PL-012 (Marketplace)

In [13]:
i = next(
    k
    for k, m in enumerate(matches)
    if m.plan_line_id == "PL-012" and fb_lines[k].service_month == "2026-01"
)
pay, ln = pays[i], fb_lines[i]
biggest = max(pay.publisher_views, pay.ad_server_views)
show(
    vz.bars(
        [
            ("Facebook counted", pay.publisher_views, "bad"),
            ("GroupM ad server counted", pay.ad_server_views, "good"),
        ],
        biggest,
    ),
    f"<p>Facebook's count is <b>{pay.publisher_above_ad_server_percent:.1f}% higher</b>; "
    f"the limit is {fb.allowed_difference_percent}%. So we pay on the ad server's count.</p>",
    vz.tiles(
        [
            ("Facebook billed", vz.usd(ln.amount), "bad"),
            ("Should pay", vz.usd(pay.expected_amount), "good"),
            ("Over-billed", vz.usd(ln.amount - pay.expected_amount), "warn"),
        ]
    ),
)

## All 14 lines: billed (red) against what should be paid (green)

In [14]:
biggest = max(ln.amount for ln in fb_lines)
show(
    vz.table(
        ["bill line", "plan line", "billed", "should pay", "compare", "gap"],
        [
            [
                f"{ln.invoice_number[-4:]} #{ln.invoice_line_number}",
                m.plan_line_id,
                vz.usd(ln.amount),
                vz.usd(p.expected_amount),
                vz.money_pair(ln.amount, p.expected_amount, biggest),
                vz.badge("✓ OK", "good")
                if ln.amount == p.expected_amount
                else vz.badge(vz.usd(ln.amount - p.expected_amount), "bad"),
            ]
            for ln, m, p in zip(fb_lines, matches, pays, strict=True)
        ],
        numeric=["billed", "should pay"],
    )
)

bill line,plan line,billed,should pay,compare,gap
7731 #1,PL-009,"$46,915.38","$46,915.38",,✓ OK
7731 #2,PL-010,"$24,216.49","$24,216.49",,✓ OK
7731 #3,PL-011,"$33,454.59","$33,454.59",,✓ OK
7731 #4,PL-012,"$31,528.29","$26,718.89",,"$4,809.40"
7731 #5,PL-013,"$18,934.57","$18,934.57",,✓ OK
7731 #6,PL-014,"$22,555.45","$22,555.45",,✓ OK
7731 #7,PL-016,"$15,099.11","$15,099.11",,✓ OK
8094 #1,PL-009,"$42,523.45","$42,523.45",,✓ OK
8094 #2,PL-010,"$21,601.03",$0.00,,"$21,601.03"
8094 #3,PL-011,"$30,075.25","$30,075.25",,✓ OK


---
# Step 4.2: Find why a line is wrong
One small check per reason. The first check that fires gives the reason, **with evidence** a person can verify.

In [15]:
from invoice_recon.models import Cause
from invoice_recon.reconcile import reconcile_lines

results = reconcile_lines(fb_lines, ref)
show(
    "<b>The reasons we check for:</b><br>",
    "".join(str(vz.badge(vz.CAUSE_WORDS[c], "plain")) + " " for c in Cause if c is not Cause.NONE),
)
flagged = [r for r in results if r.cause is not Cause.NONE]
show(
    vz.tiles(
        [
            ("lines with nothing wrong", f"{len(results) - len(flagged)} ✓", "good"),
            ("lines with a problem", str(len(flagged)), "warn"),
        ]
    ),
    *[
        vz.card(
            f"Bill {r.invoice_number[-4:]} line {r.invoice_line_number} → {r.matched_plan_line_id}:"
            f" {vz.CAUSE_WORDS[r.cause]}",
            list(r.evidence),
            "warn",
        )
        for r in flagged
    ],
)

---
# Step 4.4: Decide
Each reason leads to one of four decisions. Two safety rules: **approve only if the amount is exactly right**, and
**"pay less" only if the bill is higher than it should be**. Anything else goes to a person.

In [16]:
from invoice_recon.decide import DECISION_BY_CAUSE
from invoice_recon.models import Decision

lanes = []
for d in Decision:
    causes = [vz.CAUSE_WORDS[c] for c, x in DECISION_BY_CAUSE.items() if x is d]
    lanes.append(vz.card(vz.DECISION_WORDS[d], causes, vz.DECISION_TONE[d]))
show(f'<div class="row">{"".join(lanes)}</div>')
counts = {d: sum(r.decision is d for r in results) for d in Decision}
show(
    "<b>Result for the 14 Facebook lines</b>",
    vz.stacked([(vz.DECISION_WORDS[d], n, vz.DECISION_TONE[d]) for d, n in counts.items()]),
)

---
# Step 7.1: Score against the answer key
The answer key holds the right answer for every line. **The system never reads it**: only the scoring code in `tests/` does,
after the answers are made.

In [17]:
import sys

sys.path.insert(0, str(DATA.parents[1] / "tests"))
from scoring import score_results  # test-only code: the only reader of the answer key

score = score_results(results, "invoices/facebook")
tone = "good" if score.all_four == score.lines else "warn"
show(
    vz.tiles(
        [
            ("right plan line", f"{score.matched_plan_line} / {score.lines}", tone),
            ("right reason", f"{score.cause} / {score.lines}", tone),
            ("right decision", f"{score.decision} / {score.lines}", tone),
            ("right amount", f"{score.expected_amount} / {score.lines}", tone),
        ]
    ),
    "Mismatches: " + (", ".join(score.mismatches) or "none"),
)

---
# Dashboard: the Facebook bills at a glance

In [18]:
billed = sum(r.invoiced_amount for r in results)
should = sum(r.expected_amount for r in results)
show(
    vz.tiles(
        [
            ("billed by Facebook", vz.usd(billed), "plain"),
            ("GroupM should pay", vz.usd(should), "good"),
            ("over-billed", vz.usd(billed - should), "bad"),
            ("lines approved", f"{counts[Decision.APPROVE]} of {len(results)}", "good"),
            ("lines needing action", str(len(results) - counts[Decision.APPROVE]), "warn"),
        ]
    ),
    vz.stacked([(vz.DECISION_WORDS[d], n, vz.DECISION_TONE[d]) for d, n in counts.items()]),
)

In [19]:
biggest = max(r.invoiced_amount for r in results)
rows = []
for ln, r in zip(fb_lines, results, strict=True):
    rows.append(
        [
            f"{ln.service_month}",
            ln.line_description.split(" | ")[0].replace("_", " ").replace(" 2026", ""),
            ln.line_description.split(" | ")[1],
            r.matched_plan_line_id,
            vz.usd(r.invoiced_amount),
            vz.usd(r.expected_amount),
            vz.money_pair(r.invoiced_amount, r.expected_amount, biggest),
            vz.decision_badge(r.decision),
            vz.CAUSE_WORDS[r.cause],
        ]
    )
hot = [i for i, r in enumerate(results) if r.decision is not Decision.APPROVE]
show(
    "<b>Every bill line: billed (red/blue) against what should be paid (green)</b>",
    vz.table(
        [
            "month",
            "campaign",
            "ad",
            "plan line",
            "billed",
            "should pay",
            "compare",
            "decision",
            "why",
        ],
        rows,
        numeric=["billed", "should pay"],
        hot=hot,
    ),
)

month,campaign,ad,plan line,billed,should pay,compare,decision,why
2026-01,Pepsi WildCherry Spring,FB Feed Video,PL-009,"$46,915.38","$46,915.38",,Approve,Nothing wrong
2026-01,Pepsi WildCherry Spring,FB Feed Image,PL-010,"$24,216.49","$24,216.49",,Approve,Nothing wrong
2026-01,Pepsi WildCherry Spring,In-Stream,PL-011,"$33,454.59","$33,454.59",,Approve,Nothing wrong
2026-01,Pepsi WildCherry Spring,Marketplace,PL-012,"$31,528.29","$26,718.89",,Pay less,Publisher counted too many views
2026-01,DietPepsi NewYear,FB Feed Video,PL-013,"$18,934.57","$18,934.57",,Approve,Nothing wrong
2026-01,DietPepsi NewYear,FB Feed Image,PL-014,"$22,555.45","$22,555.45",,Approve,Nothing wrong
2026-01,DietPepsi NewYear,Marketplace,PL-016,"$15,099.11","$15,099.11",,Approve,Nothing wrong
2026-02,Pepsi WildCherry Spring,FB Feed Video,PL-009,"$42,523.45","$42,523.45",,Approve,Nothing wrong
2026-02,Pepsi WildCherry Spring,FB Feed Image,PL-010,"$21,601.03",$0.00,,Dispute,Billed an order that was cancelled
2026-02,Pepsi WildCherry Spring,In-Stream,PL-011,"$30,075.25","$30,075.25",,Approve,Nothing wrong


In [20]:
show(
    vz.card(
        "Compared with the contract",
        [
            f"Facebook bills on its own view count, within {tol}% of the ad server",
            "Extra views beyond the month's plan are not paid",
            "A cancelled order is not paid at all",
        ],
        "info",
    ),
    *[
        vz.card(
            f"Action: {vz.DECISION_WORDS[r.decision]} · {vz.usd(r.difference)} over · "
            f"{vz.CAUSE_WORDS[r.cause]}",
            list(r.evidence),
            vz.DECISION_TONE[r.decision],
        )
        for r in results
        if r.decision is not Decision.APPROVE
    ],
)

---
**Milestone 1 is finished.** Next: Milestone 2, reading all 10 bills (PDF and Excel too).